# Projet Kaggle

[Solar power generation](https://www.kaggle.com/code/pythonafroz/solar-power-generation-forecast/input?scriptVersionId=191668971&select=Plant_1_Weather_Sensor_Data.csv)

In [ ]:
import pandas as pd

source_data_path = "plant_generation_data_groupe_3"
generation_data = pd.read_csv(f'data/train_{source_data_path}.csv')
weather_data = pd.read_csv('data/Plant_Weather_Sensor_Data.csv')

In [ ]:
generation_data

# Exploration des données

In [ ]:
generation_data = generation_data.rename(
    columns={
        "DATE_TIME": "date_time",
        "PLANT_ID": "plant_id",
        "SOURCE_KEY": "source_key",
        "DC_POWER": "dc_power",
        "AC_POWER": "ac_power",
        "DAILY_YIELD": "daily_yield",
        "TOTAL_YIELD": "total_yield",
    }
)

generation_data["date_time"] = pd.to_datetime(generation_data["date_time"])
generation_data_gr = generation_data.groupby(
    by=["date_time", "plant_id"],
    as_index=False,
).agg(
    {
        "dc_power": "sum",
        "ac_power": "sum",
    }
)

In [ ]:
generation_data_gr.date_time.min()

In [ ]:
generation_data_gr.date_time.max()

In [ ]:
weather_data = weather_data.rename(
    columns={
        "DATE_TIME": "date_time",
        "PLANT_ID": "plant_id",
        "SOURCE_KEY": "source_key",
        "AMBIENT_TEMPERATURE": "ambient_temperature",
        "MODULE_TEMPERATURE": "module_temperature",
        "IRRADIATION": "irradiation",
    }
)
weather_data["date_time"] = pd.to_datetime(weather_data["date_time"])
weather_data_gr = weather_data.groupby(
    by=["date_time", "plant_id"],
    as_index=False,
).agg(
    {
        "ambient_temperature": "mean",
        "module_temperature": "mean",
        "irradiation": "mean",
    }
)

weather_data_gr.to_csv(
    'data/plant_weather_data.csv',
    index=False,
)

In [ ]:
df = generation_data_gr.merge(
    weather_data_gr,
    how="inner",
    on=["date_time", "plant_id"],
)

In [ ]:
df = df.dropna(how="any")
df.shape

In [ ]:
df.plot(
    x="date_time",
    y=["dc_power", "ac_power", "ambient_temperature", "module_temperature", "irradiation"],
    subplots=True,
    layout=(3, 2),
    figsize=(15, 10),
    title="Generation and Weather Data",
)

In [ ]:
# Group 1 : predict ac_power
# Group 2 : predict dc_power
target_pred = "dc_power"

# Feature Engineering

In [ ]:
def create_features(df):
    """
    Create features for the model
    """
    # Create a new column with the date and time
    df["date_time"] = pd.to_datetime(df["date_time"])

    # Extract date and time features
    df["day"] = df["date_time"].dt.day
    df["hour"] = df["date_time"].dt.hour

    # Extract day of week and day of year
    df["day_of_week"] = df["date_time"].dt.dayofweek
    
    # is_day
    df["is_day"] = (df["date_time"].dt.hour >= 6) & (df["date_time"].dt.hour < 18)

    return df

# Calendar features
df = create_features(df=df)

In [ ]:
df_featured = df.dropna(how="any")
df_featured.shape

In [ ]:
df_featured.head()

# Feature importance

In [ ]:
X = df_featured.drop(
    ["date_time", "plant_id", "dc_power", "ac_power"],
    axis=1,
)

y = df_featured[target_pred]     

In [ ]:
X.shape

In [ ]:
y.head()

In [ ]:
from sklearn.model_selection import train_test_split
# Import a General Regression Model
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt
import numpy as np

# Feature Importance
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

importances = model.feature_importances_
feature_names = X.columns
indices = np.argsort(importances)[::-1]
plt.figure(figsize=(10, 6))
plt.title("Feature Importances")
plt.bar(range(X.shape[1]), importances[indices], align="center")
plt.xticks(range(X.shape[1]), feature_names[indices], rotation=90)
plt.xlim([-1, X.shape[1]])
plt.show()


In [ ]:
importances_limit = 10**-4

In [ ]:
# On confirme que l'on sélectionne les features les plus corrélées
feature_selected = [
    feature_names[i] for i in indices if importances[i] > importances_limit
]
feature_selected

In [ ]:
# Keep only the 4 important features
X = X[feature_selected].dropna(how="any")

In [ ]:
X

In [ ]:
# Afficher le tableau des corrélations
import seaborn as sns
import matplotlib.pyplot as plt

correlation_matrix = X.corr()
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=True)

In [ ]:
most_important_features = feature_selected
most_important_features.remove("module_temperature")

X = X[most_important_features]

In [ ]:
X.head()

In [ ]:
def create_final_dataset(df, feature_selected):
    """
    Create the final dataset for the model
    """
    return df[feature_selected]

In [ ]:
X = create_final_dataset(df, feature_selected)

# Modélisation

In [ ]:
# Train the model linear regression
from sklearn.linear_model import LinearRegression

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

# R2: R2 est le coefficient de détermination, qui mesure la proportion de la variance des prix expliquée par le modèle
# Plus R2 est proche de 1, meilleur est le modèle
r2_score = model.score(X_test, y_test)
print(f"R² score: {r2_score:.2f}")

# MAE: L'erreur absolue moyenne (MAE) mesure la moyenne des erreurs absolues entre les valeurs prédites et réelles
# Plus la MAE est faible, meilleur est le modèle
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE: {mae:.2f}")

# RMSE: La racine carrée de l'erreur quadratique moyenne (RMSE) mesure la moyenne des erreurs au carré entre les valeurs
# prédites et réelles
# Plus la RMSE est faible, meilleur est le modèle
rmse = mean_squared_error(y_test, y_pred)
print(f"RMSE: {rmse:.2f}")


In [ ]:
# Plot with plotly y_pred & y_test
import plotly.express as px
import plotly.graph_objects as go

# 2 lines: x = datetime, y = y_test and y_pred
fig = go.Figure()

# Add y_test line
fig.add_trace(go.Scatter(
    x=X_test.index,
    y=y_test,
    mode='lines',
    name='y_test',
    line=dict(color='blue')
))

# Add y_pred line
fig.add_trace(go.Scatter(
    x=X_test.index,
    y=y_pred,
    mode='lines',
    name='y_pred',
    line=dict(color='red')
))

# Update layout
fig.update_layout(
    title="y_test vs y_pred",
    xaxis_title="Index",
    yaxis_title="Value",
    legend_title="Legend"
)

fig.show()

In [ ]:
# Train the model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)
model = RandomForestRegressor(
    n_estimators=100, random_state=42, min_samples_leaf=10, max_features=0.5)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

# R2: R2 est le coefficient de détermination, qui mesure la proportion de la variance des prix expliquée par le modèle
# Plus R2 est proche de 1, meilleur est le modèle
r2_score = model.score(X_test, y_test)
print(f"R² score: {r2_score:.2f}")

# MAE: L'erreur absolue moyenne (MAE) mesure la moyenne des erreurs absolues entre les valeurs prédites et réelles
# Plus la MAE est faible, meilleur est le modèle
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE: {mae:.2f}")

# RMSE: La racine carrée de l'erreur quadratique moyenne (RMSE) mesure la moyenne des erreurs au carré entre les valeurs
# prédites et réelles
# Plus la RMSE est faible, meilleur est le modèle
rmse = mean_squared_error(y_test, y_pred)
print(f"RMSE: {rmse:.2f}")


In [ ]:
# Plot with plotly y_pred & y_test
import plotly.express as px
import plotly.graph_objects as go

# 2 lines: x = datetime, y = y_test and y_pred
fig = go.Figure()

# Add y_test line
fig.add_trace(go.Scatter(
    x=X_test.index,
    y=y_test,
    mode='lines',
    name='y_test',
    line=dict(color='blue')
))

# Add y_pred line
fig.add_trace(go.Scatter(
    x=X_test.index,
    y=y_pred,
    mode='lines',
    name='y_pred',
    line=dict(color='red')
))

# Update layout
fig.update_layout(
    title="y_test vs y_pred",
    xaxis_title="Index",
    yaxis_title="Value",
    legend_title="Legend"
)

fig.show()

In [ ]:
df_to_pred = pd.read_csv(f'data/test_{source_data_path}.csv')
df_to_pred["date_time"] = pd.to_datetime(df_to_pred["date_time"])
df_to_pred = df_to_pred.merge(
    weather_data_gr,
    how="inner",
    on=["date_time", "plant_id"],
)

df_to_pred_features = create_features(df=df_to_pred)
df_to_pred_selected = create_final_dataset(
    df=df_to_pred_features,
    feature_selected=feature_selected,
)
df_to_pred_selected.head()

In [ ]:
prevision = model.predict(df_to_pred_selected)
df_to_pred_selected["dc_power_prevision"] = prevision
df_to_pred_selected["date_time"] = df_to_pred["date_time"]

# Visualisation des résultats

In [ ]:
# Plot with plotly y_pred & y_test
import plotly.express as px
import plotly.graph_objects as go

# 2 lines: x = datetime, y = y_test and y_pred
fig = go.Figure()

# Add y_test line
fig.add_trace(go.Scatter(
    x=df_to_pred_selected.date_time,
    y=df_to_pred_selected.dc_power_prevision,
    mode='lines',
    name='données réelles',
    line=dict(color='blue')
))

# Update layout
fig.update_layout(
    title="Prévision de la production d'énergie photovoltaïque (Random Forest)",
    xaxis_title="Date",
    yaxis_title="Valeurs",
    legend_title="Legend"
)

fig.show()

# Bonus

In [ ]:
df_pred_ac = df.drop(
    ["date_time", "plant_id", "ac_power"],
    axis=1,
)
y = df["ac_power"]

In [ ]:
# Train random forest model
model = RandomForestRegressor(
    n_estimators=100, random_state=42, min_samples_leaf=10, max_features=0.5)
model.fit(df_pred_ac, y)

In [ ]:
df_to_pred["dc_power"] = prevision
df_to_pred_ac = df_to_pred[df_pred_ac.columns]

In [ ]:
df_to_pred_ac["ac_power"] = model.predict(df_to_pred_ac)

In [ ]:
df_to_pred_ac["efficacite_onduleur"] = df_to_pred_ac["ac_power"] / df_to_pred_ac["dc_power"]
df_to_pred_ac["efficacite_onduleur"].describe()

In [ ]:
df_to_pred_ac.head()

In [ ]:
# Plot with plotly y_pred & y_test
import plotly.express as px
import plotly.graph_objects as go

# 2 lines: x = datetime, y = y_test and y_pred
fig = go.Figure()

# Add dc_power line
fig.add_trace(go.Scatter(
    x=df_to_pred_ac.index,
    y=df_to_pred_ac.dc_power,
    mode='lines',
    name='dc_power',
    line=dict(color='red')
))

# Add ac_power line
fig.add_trace(go.Scatter(
    x=df_to_pred_ac.index,
    y=df_to_pred_ac.ac_power,
    mode='lines',
    name='ac_power',
    line=dict(color='green')
))

# Add efficacite_onduleur line on a secondary y-axis
fig.add_trace(go.Scatter(
    x=df_to_pred_ac.index,
    y=df_to_pred_ac.efficacite_onduleur,
    mode='lines',
    name='efficacité onduleur',
    line=dict(color='blue'),
    yaxis='y2'
))

# Update layout to include a secondary y-axis
fig.update_layout(
    title="Prévision de la production d'énergie photovoltaïque (Random Forest)",
    xaxis_title="Date",
    yaxis_title="Valeurs",
    yaxis2=dict(
        title="Efficacité Onduleur",
        overlaying='y',
        side='right'
    ),
    legend_title="Legend"
)
fig.show()

In [ ]:
# Afficher la distribution de la variable
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.histplot(df_to_pred_ac["efficacite_onduleur"], bins=50, kde=True)
plt.title("Distribution de l'efficacité de l'onduleur")
plt.xlabel("Efficacité de l'onduleur")
plt.ylabel("Fréquence")
plt.show()

In [ ]:
# Visualisation des résidus
residuals = y_test - y_pred
plt.figure(figsize=(10, 6))
plt.scatter(y_test, residuals)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Valeurs réelles')
plt.ylabel('Résidus')
plt.title('Résidus vs Valeurs réelles')
plt.show()

In [ ]:
# Distribution des résidus
plt.figure(figsize=(10, 6))
plt.hist(residuals, bins=150, edgecolor='k', alpha=0.7)
plt.xlabel('Résidus')
plt.ylabel('Fréquence')
plt.title('Distribution des résidus')
plt.show()

In [ ]:
# Sont ils normalement distribués ?
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Q-Q plot
plt.figure(figsize=(10, 6))
stats.probplot(residuals, dist="norm", plot=plt)
plt.title('Q-Q plot des résidus')
plt.show()



In [ ]:
# Test de normalité de Shapiro-Wilk
shapiro_test = stats.shapiro(residuals)
shapiro_test_statistic = shapiro_test.statistic
shapiro_test_p_value = shapiro_test.pvalue
alpha = 0.05
if shapiro_test_p_value > alpha:
    print("Les résidus suivent une distribution normale (H0 acceptée)")
else:
    print("Les résidus ne suivent pas une distribution normale (H0 rejetée)")

In [ ]:
shapiro_test_p_value

In [ ]:
# Il manque donc des variables explicatives pour expliquer la production d'énergie photovoltaïque